In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Load data
file_path = "Figure_6K_TSC22D2_Motif-deletion-mutants_Foci_Size.csv"
df = pd.read_csv(file_path)

# Reshape to long format
df_long = df.melt(var_name='Condition', value_name='Foci Size')
df_long['Osmolarity'] = df_long['Condition'].apply(lambda x: "300_mOsm_per_L" if "300" in x else "700_mOsm_per_L")

# Colors for violins
violin_colors = {"300_mOsm_per_L": "#e8e8e8", "700_mOsm_per_L": "#49c1bb"}
condition_order = df.columns.tolist()  # Ensure consistent order

plt.rcParams['font.family'] = 'Arial'
fig, ax = plt.subplots(figsize=(4, 3.2))

# --- Violin plot (full first) ---
vp = sns.violinplot(
    x='Condition', y='Foci Size', data=df_long,
    hue='Osmolarity', order=condition_order,
    inner=None, cut=0, scale='width', palette=violin_colors,
    linewidth=0, dodge=False, ax=ax
)

# --- Clip violins to left half only ---
for violin in ax.collections:
    path = violin.get_paths()[0]
    vertices = path.vertices
    center_x = np.median(vertices[:, 0])  # Find center x
    vertices[:, 0] = np.minimum(vertices[:, 0], center_x)  # Clip to left half
    path.vertices = vertices

# --- Boxplot (white fill) ---
sns.boxplot(
    x='Condition', y='Foci Size', hue='Osmolarity', data=df_long,
    order=condition_order, showcaps=False, showfliers=False, width=0.35,
    boxprops={'facecolor': 'white', 'edgecolor': 'black', 'zorder': 4},
    medianprops={'color': 'black', 'linewidth': 1.2},
    whiskerprops={'color': 'black'}, capprops={'color': 'black'},
    palette=['white'], dodge=False, ax=ax
)

# --- Stripplot (white dots with black edge) ---
sns.stripplot(
    x='Condition', y='Foci Size', hue='Osmolarity', data=df_long,
    order=condition_order, jitter=True, size=3.5, alpha=1,
    palette=['white'], edgecolor='black', linewidth=0.6,
    dodge=False, ax=ax
)

# --- Mean as horizontal line ---
group_means = df_long.groupby('Condition')['Foci Size'].mean()
for i, cond in enumerate(condition_order):
    ax.hlines(
        y=group_means[cond], xmin=i - 0.2, xmax=i + 0.2,
        colors='black', linewidth=1.2, zorder=6
    )

# Remove legends
if ax.get_legend():
    ax.get_legend().remove()

# Vertical group separators
for pos in [3.5]:
    ax.axvline(x=pos, color='black', linestyle='--', linewidth=1)

# Labels
plt.xlabel("Condition")
plt.ylabel("Foci Size (µm²)")
plt.xticks(rotation=45, ha='right')

# --- Y-axis range ---
ax.set_ylim(top=500)

# Save
plt.savefig("Figure_6K_TSC22D2_Motif-deletion-mutants.pdf", format="pdf", dpi=300, bbox_inches="tight", transparent=True)
plt.show()


In [ ]:
import pandas as pd
import scipy.stats as stats
from itertools import combinations
from statsmodels.stats.multitest import multipletests

# --- Load data ---
file_path = "Figure_6K_TSC22D2_Motif-deletion-mutants_Foci_Size.csv"
df = pd.read_csv(file_path)

# --- Reshape to long format ---
df_long = df.melt(var_name="Condition", value_name="Foci_Count").dropna()
df_long["Construct"] = df_long["Condition"].apply(lambda x: "FL" if "FL" in x else ("delTbrN" if "delTbrN" in x else ("delFQ" if "delFQ" in x else "delRphi1")))
df_long["Source"] = df_long["Condition"].apply(lambda x: "300_mOsm_per_L" if "300_mOsm_per_L" in x else ("700_mOsm_per_L"))
df_long["Group"] = df_long["Construct"] + "_" + df_long["Source"]

# --- Prepare groups ---
groups = df_long.groupby("Group")["Foci_Count"].apply(list)
pairs = list(combinations(groups.index, 2))

# --- Mann–Whitney U tests ---
results = []
for g1, g2 in pairs:
    u_stat, p_val = stats.mannwhitneyu(groups[g1], groups[g2], alternative='two-sided')
    results.append({"Group1": g1, "Group2": g2, "U_stat": u_stat, "p_uncorrected": p_val})

results_df = pd.DataFrame(results)

# --- FDR correction (Benjamini-Hochberg) ---
reject, p_corrected, _, _ = multipletests(results_df["p_uncorrected"], method='fdr_bh')
results_df["p_corrected_FDR"] = p_corrected
results_df["Significant"] = reject

# --- Sort by corrected p-value ---
results_df = results_df.sort_values("p_corrected_FDR")

# --- Save or display ---
print(results_df)